In [1]:
import os
import pandas as pd
import xarray as xr

DATA_DIR = "data_extracted"

all_years = []

for year_folder in sorted(os.listdir(DATA_DIR)):

    folder_path = os.path.join(DATA_DIR, year_folder)

    if not os.path.isdir(folder_path):
        continue

    nc_files = [f for f in os.listdir(folder_path) if f.endswith(".nc")]

    if len(nc_files) == 0:
        print(f"No .nc file in {year_folder}")
        continue

    nc_path = os.path.join(folder_path, nc_files[0])

    year = int(year_folder.split("_")[-1])

    print(f"Processing {year}")

    ds = xr.open_dataset(
        nc_path,
        engine="netcdf4"
    )

    temp = ds["t2m"] - 273.15
    precip = ds["tp"]
    snow = ds["sf"]

    sep_temp = temp.sel(
        valid_time=temp.valid_time.dt.month == 9
    ).mean("valid_time")

    oct_temp = temp.sel(
        valid_time=temp.valid_time.dt.month == 10
    ).mean("valid_time")

    nov_temp = temp.sel(
        valid_time=temp.valid_time.dt.month == 11
    ).mean("valid_time")

    sep_precip = precip.sel(
        valid_time=precip.valid_time.dt.month == 9
    ).sum("valid_time")

    oct_precip = precip.sel(
        valid_time=precip.valid_time.dt.month == 10
    ).sum("valid_time")

    nov_precip = precip.sel(
        valid_time=precip.valid_time.dt.month == 11
    ).sum("valid_time")

    winter_snowfall = snow.sel(
        valid_time=snow.valid_time.dt.month.isin([12,1,2,3,4])
    ).sum("valid_time")

    df = pd.DataFrame({
        "lat": sep_temp.latitude.values.repeat(len(sep_temp.longitude)),
        "lon": list(sep_temp.longitude.values) * len(sep_temp.latitude),

        "sep_temp": sep_temp.values.flatten(),
        "oct_temp": oct_temp.values.flatten(),
        "nov_temp": nov_temp.values.flatten(),

        "sep_precip": sep_precip.values.flatten(),
        "oct_precip": oct_precip.values.flatten(),
        "nov_precip": nov_precip.values.flatten(),

        "winter_snowfall": winter_snowfall.values.flatten()
    })

    df["year"] = year

    all_years.append(df)

final_df = pd.concat(all_years, ignore_index=True)

print(final_df.shape)

final_df.to_csv(
    "alps_snowfall_ml_dataset.csv",
    index=False
)

print("Saved.")

Processing 2000
Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
Processing 2025
(147186, 10)
Saved.
